# Experiment 1: Adult Dataset Exploratory Data Analysis (EDA)
**Date**: 2025-12-16  
**Author**: PhD Candidate

## 1. Overview
This notebook provides a comprehensive analysis of the UCI Adult dataset used in Experiment 1. 
We investigate:
- Class imbalance ratios
- Feature distributions (Categorical vs Numerical)
- Correlation structures

**Goal**: validate that the dataset properties match those reported in literature and identify potential biases that could affect XAI method fidelity.

### 2. Setup and Data Loading
First, we initialize the environment and load the cached dataset using our framework's data loader. 
This ensures we are analyzing the exact same data version used for model training.

In [ ]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Add project root to path
sys.path.append(os.path.abspath("../../.."))

from src.data_loading.adult import load_adult

# Load data (returns X_train, X_test, y_train, y_test)
X_train, X_test, y_train, y_test = load_adult()

# Reconstruct DataFrame for EDA
df = pd.concat([X_train, y_train], axis=1)
print(f"Dataset Shape: {df.shape}")
df.head()

### 3. Class Imbalance Analysis
We examine the target variable (`income`) distribution. Extreme imbalance can skew model performance metrics (e.g., Accuracy) and affect the baseline for counterfactual generation.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x=y_train)
plt.title("Target Distribution (Income > 50K)")
plt.xlabel("Class (0: <=50K, 1: >50K)")
plt.ylabel("Count")
plt.show()

ratio = sum(y_train) / len(y_train)
print(f"Positive Class Ratio: {ratio:.2%}")

**Interpretation**:
The dataset shows a ~24% positive class rate. This confirms the known 1:3 imbalance. 
**Implication**: We must use `class_weight='balanced'` in our Random Forest training (as configured in `rf_adult_config.yaml`).

### 4. Correlation Analysis
We visualize the correlation matrix to identify collinear features which might destabilize feature importance explanations (e.g., LIME).

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', center=0)
plt.title("Feature Correlation Matrix")
plt.show()

## 5. Conclusions
1.  **Integrity**: Data loaded successfully with expected dimensions.
2.  **Imbalance**: Significant (24% positive). Metrics like F1-Score and AUC are preferred over Accuracy.
3.  **Features**: High dimensionality after One-Hot Encoding requires efficient explainers.

## 6. References
- [1] Duo, W. and Vlahu, H. "UCI Machine Learning Repository: Adult Data Set". 1996.
- [2] Project Documentation: `experiments/exp1_adult/README.md`